### Set explicit data directory and load pyspark

In [57]:
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
DATA_DIR = PROJECT_ROOT / "data"

In [58]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("data-modeling-kimball")
    .getOrCreate()
)

print(spark.version)

3.5.4


## Kimball Model

Business process: Sales

Grain:
One row per product per order.

Dimensions:
- Customer
- Product
- Date

Fact:
- Sales

### Create customers silver

In [59]:
bronze_customers = spark.read.parquet(
    str(DATA_DIR / "bronze" / "crm_customers")
)

bronze_customers.show()

+-----------+----------+---------+--------------------+-----------+-----+-------------------+
|customer_id|first_name|last_name|               email|       city|state|         created_at|
+-----------+----------+---------+--------------------+-----------+-----+-------------------+
|       1002|    Marcus|     Hill|marcus.hill@examp...|     Austin|   TX|2026-02-03 14:10:00|
|       1003|     Priya|     Shah|priya.shah@exampl...|    Seattle|   WA|2026-02-20 11:45:00|
|       1004|    Daniel|      Kim|daniel.kim@exampl...|    Chicago|   IL|2026-03-08 16:20:00|
|       1001|     Alice|   Nguyen|alice.nguyen@exam...|Los Angeles|   CA|2026-01-15 09:30:00|
+-----------+----------+---------+--------------------+-----------+-----+-------------------+



In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

customer_window = Window.orderBy("customer_id")

dim_customer = (
    bronze_customers
    .withColumn(
        "customer_key",
        F.row_number().over(customer_window)
    )
    .select(
        "customer_key",
        "customer_id",
        "first_name",
        "last_name",
        "email",
        "city",
        "state",
        "created_at"
    )
)

dim_customer.show()

In [ ]:
dim_customer.write.mode("overwrite").parquet(
    str(DATA_DIR / "kimball" / "dim_customer")
)

In [62]:
spark.read.parquet(
    str(DATA_DIR / "kimball" / "dim_customer")
).show()

+------------+-----------+----------+---------+--------------------+-----------+-----+-------------------+
|customer_key|customer_id|first_name|last_name|               email|       city|state|         created_at|
+------------+-----------+----------+---------+--------------------+-----------+-----+-------------------+
|           1|       1001|     Alice|   Nguyen|alice.nguyen@exam...|Los Angeles|   CA|2026-01-15 09:30:00|
|           2|       1002|    Marcus|     Hill|marcus.hill@examp...|     Austin|   TX|2026-02-03 14:10:00|
|           3|       1003|     Priya|     Shah|priya.shah@exampl...|    Seattle|   WA|2026-02-20 11:45:00|
|           4|       1004|    Daniel|      Kim|daniel.kim@exampl...|    Chicago|   IL|2026-03-08 16:20:00|
+------------+-----------+----------+---------+--------------------+-----------+-----+-------------------+



### Create Products Silver

In [63]:
bronze_products = spark.read.parquet(
    str(DATA_DIR / "bronze" / "erp_products")
)

bronze_products.show()

+----------+-------------------+-----------+---------+----------+
|product_id|       product_name|   category|unit_cost|unit_price|
+----------+-------------------+-----------+---------+----------+
|      2001|     Wireless Mouse|Electronics|    18.00|     34.99|
|      2002|Mechanical Keyboard|Electronics|    52.00|     89.99|
|      2003|       Laptop Stand|     Office|    21.50|     44.99|
|      2004|          USB-C Hub|Electronics|    28.00|     54.99|
+----------+-------------------+-----------+---------+----------+



In [ ]:
product_window = Window.orderBy("product_id")

dim_product = (
    bronze_products
    .withColumn(
        "product_key",
        F.row_number().over(product_window)
    )
    .select(
        "product_key",
        "product_id",
        "product_name",
        "category",
        "unit_cost",
        "unit_price"
    )
)

dim_product.show()

In [ ]:
dim_product.write.mode("overwrite").parquet(
    str(DATA_DIR / "kimball" / "dim_product")
)

In [66]:
spark.read.parquet(
    str(DATA_DIR / "kimball" / "dim_product")
).show()

+-----------+----------+-------------------+-----------+---------+----------+
|product_key|product_id|       product_name|   category|unit_cost|unit_price|
+-----------+----------+-------------------+-----------+---------+----------+
|          1|      2001|     Wireless Mouse|Electronics|    18.00|     34.99|
|          2|      2002|Mechanical Keyboard|Electronics|    52.00|     89.99|
|          3|      2003|       Laptop Stand|     Office|    21.50|     44.99|
|          4|      2004|          USB-C Hub|Electronics|    28.00|     54.99|
+-----------+----------+-------------------+-----------+---------+----------+



### Create dim_date silver

In [67]:
from datetime import date

start_date = date(2026, 1, 1)
end_date = date(2026, 12, 31)

dim_date = (
    spark.sql(
        f"""
        SELECT explode(
            sequence(
                to_date('{start_date}'),
                to_date('{end_date}'),
                interval 1 day
            )
        ) AS full_date
        """
    )
    .withColumn("date_key", F.date_format("full_date", "yyyyMMdd").cast("int"))
    .withColumn("year", F.year("full_date"))
    .withColumn("month", F.month("full_date"))
    .withColumn("day", F.dayofmonth("full_date"))
    .withColumn("month_name", F.date_format("full_date", "MMMM"))
    .withColumn("day_name", F.date_format("full_date", "EEEE"))
    .select(
        "date_key",
        "full_date",
        "year",
        "month",
        "day",
        "month_name",
        "day_name"
    )
)

In [68]:
print(dim_date.count())

dim_date.orderBy("full_date").show(5)
dim_date.orderBy(F.desc("full_date")).show(5)

365
+--------+----------+----+-----+---+----------+--------+
|date_key| full_date|year|month|day|month_name|day_name|
+--------+----------+----+-----+---+----------+--------+
|20260101|2026-01-01|2026|    1|  1|   January|Thursday|
|20260102|2026-01-02|2026|    1|  2|   January|  Friday|
|20260103|2026-01-03|2026|    1|  3|   January|Saturday|
|20260104|2026-01-04|2026|    1|  4|   January|  Sunday|
|20260105|2026-01-05|2026|    1|  5|   January|  Monday|
+--------+----------+----+-----+---+----------+--------+
only showing top 5 rows

+--------+----------+----+-----+---+----------+---------+
|date_key| full_date|year|month|day|month_name| day_name|
+--------+----------+----+-----+---+----------+---------+
|20261231|2026-12-31|2026|   12| 31|  December| Thursday|
|20261230|2026-12-30|2026|   12| 30|  December|Wednesday|
|20261229|2026-12-29|2026|   12| 29|  December|  Tuesday|
|20261228|2026-12-28|2026|   12| 28|  December|   Monday|
|20261227|2026-12-27|2026|   12| 27|  December|   Su

In [69]:
dim_date.write.mode("overwrite").parquet(
    str(DATA_DIR / "kimball" / "dim_date")
)

In [70]:
spark.read.parquet(
    str(DATA_DIR / "kimball" / "dim_date")
).orderBy("full_date").show(5)

+--------+----------+----+-----+---+----------+--------+
|date_key| full_date|year|month|day|month_name|day_name|
+--------+----------+----+-----+---+----------+--------+
|20260101|2026-01-01|2026|    1|  1|   January|Thursday|
|20260102|2026-01-02|2026|    1|  2|   January|  Friday|
|20260103|2026-01-03|2026|    1|  3|   January|Saturday|
|20260104|2026-01-04|2026|    1|  4|   January|  Sunday|
|20260105|2026-01-05|2026|    1|  5|   January|  Monday|
+--------+----------+----+-----+---+----------+--------+
only showing top 5 rows



### Write fact orders silver

In [71]:
bronze_orders = spark.read.parquet(
    str(DATA_DIR / "bronze" / "sales_orders")
)

bronze_order_items = spark.read.parquet(
    str(DATA_DIR / "bronze" / "sales_order_items")
)

In [72]:
sales_lines = (
    bronze_order_items
    .join(
        bronze_orders,
        on="order_id",
        how="inner"
    )
)

sales_lines.show()

+--------+----------+--------+----------+-----------+----------+------------+------------+
|order_id|product_id|quantity|unit_price|customer_id|order_date|order_status|total_amount|
+--------+----------+--------+----------+-----------+----------+------------+------------+
|    5001|      2002|       1|     89.99|       1001|2026-03-10|   Completed|      129.99|
|    5001|      2001|       1|     34.99|       1001|2026-03-10|   Completed|      129.99|
|    5002|      2003|       2|     44.99|       1002|2026-03-11|   Completed|       84.50|
|    5003|      2004|       1|     54.99|       1001|2026-03-14|     Pending|       42.00|
|    5004|      2002|       2|     89.99|       1004|2026-03-18|   Completed|      215.75|
|    5005|      2001|       1|     34.99|       1003|2026-03-20|   Cancelled|       59.99|
+--------+----------+--------+----------+-----------+----------+------------+------------+



In [ ]:
fact_sales = (
    sales_lines
    .join(
        dim_customer.select("customer_key", "customer_id"),
        on="customer_id",
        how="left"
    )
    .join(
        dim_product.select("product_key", "product_id"),
        on="product_id",
        how="left"
    )
    .join(
        dim_date.select("date_key", "full_date"),
        sales_lines.order_date == dim_date.full_date,
        how="left"
    )
    .withColumn(
        "line_amount",
        F.col("quantity") * F.col("unit_price")
    )
    .select(
        "order_id",
        "date_key",
        "customer_key",
        "product_key",
        "quantity",
        "unit_price",
        "line_amount",
        "order_status"
    )
)

fact_sales.show()

In [ ]:
fact_sales.write.mode("overwrite").parquet(
    str(DATA_DIR / "kimball" / "fact_sales")
)

In [75]:
spark.read.parquet(
    str(DATA_DIR / "kimball" / "fact_sales")
).show()

+--------+--------+------------+-----------+--------+----------+-----------+------------+
|order_id|date_key|customer_key|product_key|quantity|unit_price|line_amount|order_status|
+--------+--------+------------+-----------+--------+----------+-----------+------------+
|    5001|20260310|           1|          1|       1|     34.99|      34.99|   Completed|
|    5001|20260310|           1|          2|       1|     89.99|      89.99|   Completed|
|    5002|20260311|           2|          3|       2|     44.99|      89.98|   Completed|
|    5003|20260314|           1|          4|       1|     54.99|      54.99|     Pending|
|    5004|20260318|           4|          2|       2|     89.99|     179.98|   Completed|
|    5005|20260320|           3|          1|       1|     34.99|      34.99|   Cancelled|
+--------+--------+------------+-----------+--------+----------+-----------+------------+



### Exploring stable surrogate key on dim_customer

In [76]:
existing_dim_customer = spark.read.parquet(
    str(DATA_DIR / "kimball" / "dim_customer")
)

existing_dim_customer.show()

+------------+-----------+----------+---------+--------------------+-----------+-----+-------------------+
|customer_key|customer_id|first_name|last_name|               email|       city|state|         created_at|
+------------+-----------+----------+---------+--------------------+-----------+-----+-------------------+
|           1|       1001|     Alice|   Nguyen|alice.nguyen@exam...|Los Angeles|   CA|2026-01-15 09:30:00|
|           2|       1002|    Marcus|     Hill|marcus.hill@examp...|     Austin|   TX|2026-02-03 14:10:00|
|           3|       1003|     Priya|     Shah|priya.shah@exampl...|    Seattle|   WA|2026-02-20 11:45:00|
|           4|       1004|    Daniel|      Kim|daniel.kim@exampl...|    Chicago|   IL|2026-03-08 16:20:00|
+------------+-----------+----------+---------+--------------------+-----------+-----+-------------------+



In [77]:
from datetime import datetime

new_customer = spark.createDataFrame(
    [
        (
            1005,
            "Sofia",
            "Martinez",
            "sofia.martinez@example.com",
            "Denver",
            "CO",
            datetime(2026, 4, 1, 10, 0, 0)
        )
    ],
    [
        "customer_id",
        "first_name",
        "last_name",
        "email",
        "city",
        "state",
        "created_at"
    ]
)

new_customer.show()

+-----------+----------+---------+--------------------+------+-----+-------------------+
|customer_id|first_name|last_name|               email|  city|state|         created_at|
+-----------+----------+---------+--------------------+------+-----+-------------------+
|       1005|     Sofia| Martinez|sofia.martinez@ex...|Denver|   CO|2026-04-01 10:00:00|
+-----------+----------+---------+--------------------+------+-----+-------------------+



In [78]:
max_customer_key = (
    existing_dim_customer
    .agg(F.max("customer_key").alias("max_key"))
    .collect()[0]["max_key"]
)

print(max_customer_key)

4


In [79]:
new_customer_with_key = (
    new_customer
    .withColumn(
        "customer_key",
        F.lit(max_customer_key + 1)
    )
    .select(
        "customer_key",
        "customer_id",
        "first_name",
        "last_name",
        "email",
        "city",
        "state",
        "created_at"
    )
)

new_customer_with_key.show()

+------------+-----------+----------+---------+--------------------+------+-----+-------------------+
|customer_key|customer_id|first_name|last_name|               email|  city|state|         created_at|
+------------+-----------+----------+---------+--------------------+------+-----+-------------------+
|           5|       1005|     Sofia| Martinez|sofia.martinez@ex...|Denver|   CO|2026-04-01 10:00:00|
+------------+-----------+----------+---------+--------------------+------+-----+-------------------+



In [80]:
updated_dim_customer = existing_dim_customer.unionByName(
    new_customer_with_key
)

updated_dim_customer.orderBy("customer_key").show()

+------------+-----------+----------+---------+--------------------+-----------+-----+-------------------+
|customer_key|customer_id|first_name|last_name|               email|       city|state|         created_at|
+------------+-----------+----------+---------+--------------------+-----------+-----+-------------------+
|           1|       1001|     Alice|   Nguyen|alice.nguyen@exam...|Los Angeles|   CA|2026-01-15 09:30:00|
|           2|       1002|    Marcus|     Hill|marcus.hill@examp...|     Austin|   TX|2026-02-03 14:10:00|
|           3|       1003|     Priya|     Shah|priya.shah@exampl...|    Seattle|   WA|2026-02-20 11:45:00|
|           4|       1004|    Daniel|      Kim|daniel.kim@exampl...|    Chicago|   IL|2026-03-08 16:20:00|
|           5|       1005|     Sofia| Martinez|sofia.martinez@ex...|     Denver|   CO|2026-04-01 10:00:00|
+------------+-----------+----------+---------+--------------------+-----------+-----+-------------------+



In [81]:
updated_dim_customer = (
    existing_dim_customer
    .unionByName(new_customer_with_key)
    .cache()
)

updated_dim_customer.count()

5

In [82]:
updated_dim_customer.write.mode("overwrite").parquet(
    str(DATA_DIR / "kimball" / "dim_customer")
)

In [83]:
dim_customer = spark.read.parquet(
    str(DATA_DIR / "kimball" / "dim_customer")
)

dim_customer.orderBy("customer_key").show()

+------------+-----------+----------+---------+--------------------+-----------+-----+-------------------+
|customer_key|customer_id|first_name|last_name|               email|       city|state|         created_at|
+------------+-----------+----------+---------+--------------------+-----------+-----+-------------------+
|           1|       1001|     Alice|   Nguyen|alice.nguyen@exam...|Los Angeles|   CA|2026-01-15 09:30:00|
|           2|       1002|    Marcus|     Hill|marcus.hill@examp...|     Austin|   TX|2026-02-03 14:10:00|
|           3|       1003|     Priya|     Shah|priya.shah@exampl...|    Seattle|   WA|2026-02-20 11:45:00|
|           4|       1004|    Daniel|      Kim|daniel.kim@exampl...|    Chicago|   IL|2026-03-08 16:20:00|
|           5|       1005|     Sofia| Martinez|sofia.martinez@ex...|     Denver|   CO|2026-04-01 10:00:00|
+------------+-----------+----------+---------+--------------------+-----------+-----+-------------------+

